# Lab 22: Evaluating grounded generation, and unit economics

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/prekshapraveen/Documents/lantern-lab2-submission.nosync | LLM mode: replay


In [2]:
assert Path("outputs/qa/dev.jsonl").exists(), "Run notebook 18 first: it writes outputs/qa/dev.jsonl."

In [3]:
# From the slide: The numeric-claim linter
import math, re
NUM = re.compile(r"""(?P<neg>\()?\$?\s*
  (?P<num>\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)
  \s*(?P<unit>%|x\b|bn\b|mm\b|m\b|million\b|k\b)?
  (?(neg)\))""", re.X | re.I)
YEAR = re.compile(r"^(19|20)\d{2}$")
SCALE = {"bn": 1e9, "mm": 1e6, "m": 1e6, "million": 1e6,
         "k": 1e3}

def numbers_in(text):
    for m in NUM.finditer(text):
        raw, unit = m["num"], (m["unit"] or "").lower()
        a, b = m.start("num"), m.end("num")
        if (YEAR.match(raw) and not unit
                or text[a-1:a].isalpha()   # Q2, FY24
                or re.match(r"-[A-Za-z]", text[b:b+2])
                or re.search(r"(item|note)\s*$",
                             text[max(0, a-6):a], re.I)):
            continue               # 10-K, 8-K, Item 7
        v = float(raw.replace(",", ""))
        v *= SCALE.get(unit, 1)
        yield -v if m["neg"] else v

def lint(text, allowed, tol=0.005):     # [] == pass
    ok = lambda v: any(math.isclose(v, a, rel_tol=tol)
                       for a in allowed)
    return [v for v in numbers_in(text) if not ok(v)]

In [4]:
answers = [json.loads(l) for l in open("outputs/qa/dev.jsonl")]
text = {(r["doc_id"], r["block_id"]): r["text"] for r in sd.records("deal_a")}   # block ids repeat across documents
cells = {f["citation"]["cell"]: f["value"] for f in sd.seller_facts() if "cell" in f["citation"]}
def supports(cit, value):
    if cit["block_id"] in cells: return abs(cells[cit["block_id"]] - value) <= 0.005 * abs(value)
    return any(abs(v - value) <= 0.005 * abs(value) or abs(v - value * 100) <= 0.005 * abs(value * 100) for v in numbers_in(text.get((cit["doc_id"], cit["block_id"]), "")))
cited = supported = claims_with_support = n_claims = 0
for a in answers:
    for c in a["answer"]["claims"]:
        n_claims += 1; ok = False
        for cit in c["citations"]:
            cited += 1; s = all(supports(cit, n["value"]) for n in c["numbers"]); supported += s; ok |= s
        claims_with_support += ok
print(f"citation precision {supported}/{cited} | citation recall {claims_with_support}/{n_claims}")

citation precision 3/3 | citation recall 3/3


In [5]:
import csv
from sklearn.metrics import cohen_kappa_score
items = list(csv.DictReader(open("data/eval/judge_items.csv")))
judge = {l.item_id: l.supported for l in llm.structured(schemas.JudgeLabels, prompts.judge_messages([{"item_id": r["item_id"], "claim": r["claim"], "cited_text": r["cited_text"]} for r in items]), llm.P["llm"]["models"]["judge"]).labels}
a = [int(r["human_a"]) for r in items]; b = [int(r["human_b"]) for r in items]; j = [int(judge[r["item_id"]]) for r in items]
print(f"humans agree: kappa {cohen_kappa_score(a, b):.2f} | judge vs human A: kappa {cohen_kappa_score(a, j):.2f}")

humans agree: kappa 0.78 | judge vs human A: kappa 0.89


In [6]:
# Unit economics for one cited answer, from the price table and the human baseline (x0.5, x1, x2)
import pandas as pd
hm = pd.read_csv("data/reference/human_minutes.csv").set_index("task_category").loc["cited Q&A answer"]
msgs = prompts.answer_messages(sd.dev_tasks()["dev-w-01"]["turns"][0]["content"], sd.seller_facts(), [], sd.evidence_for("dev-w-01"))
tin = sum(len(m["content"]) for m in msgs) // 4
cost = llm.usd(llm.P["llm"]["models"]["large"], tin, 300)
for mult in (0.5, 1, 2):
    saved = hm["minutes_typical"] * mult
    print(f"minutes saved x{mult}: {saved:.0f} min of a $150/h associate = ${saved / 60 * 150:.2f} vs model cost ${cost:.4f}")
Path("reports").mkdir(exist_ok=True)
json.dump({"cs2_sample": {"cost_per_cited_answer_usd": round(cost, 5), "tokens_in_est": tin, "human_minutes_typical": int(hm["minutes_typical"])}}, open("reports/unit_economics_sample.json", "w"), indent=1)

minutes saved x0.5: 6 min of a $150/h associate = $15.00 vs model cost $0.0007
minutes saved x1: 12 min of a $150/h associate = $30.00 vs model cost $0.0007
minutes saved x2: 24 min of a $150/h associate = $60.00 vs model cost $0.0007


**Replay proof.** In your CS2 repository: `LANTERN_LLM_MODE=replay dvc repro`, offline, then `dvc metrics diff` against a second run must show no change. Here, rerun this notebook: every model call is a cache hit and every number is identical.

**Exercise 22.** Find the one item where the two humans disagree. Should the judge be scored against human A, human B, or their consensus?

In [7]:
for r, x, y, z in zip(items, a, b, j):
    if x != y:
        print(r["item_id"], "A", x, "B", y, "judge", z,
              "|", r["claim"], "|", r["cited_text"])
agree = [i for i in range(len(a)) if a[i] == b[i]]
print("judge vs consensus:", cohen_kappa_score(
      [a[i] for i in agree], [j[i] for i in agree]), len(agree))

j16 A 1 B 0 judge 1 | Revenue grew about 25% in FY2025. | Revenue | 26.4 | 33.07 | 41.2
j20 A 0 B 1 judge 1 | The contract term is 24 months. | Term: 36 months
judge vs consensus: 1.0 18


### Exercise 22: answer

**Where do the humans disagree?** On 2 of the 20 items:
- **j16** "Revenue grew about 25% in FY2025", cited to the revenue row 26.4 / 33.07 / 41.2:
  A = supported, B = not supported, judge = supported.
- **j20** "The contract term is 24 months", cited to "Term: 36 months":
  A = not supported, B = supported, judge = supported.

**Rubric ambiguity or a mistake?**
- **j16 is rubric ambiguity.** FY2025 growth is 41.2 / 33.07 − 1 = 24.6%, which rounds to "about 25%".
  Whether a rounded, derived number counts as supported isn't stated in the rubric, so both labels are
  defensible. The fix is to write the convention down (for example: derived numbers are supported if
  they round to the stated value).
- **j20 is an error.** "24 months" is plainly not supported by "Term: 36 months". Human B and the judge
  both got it wrong.

**Score the judge against A, B, or their consensus?** The consensus: the 18 items where A and B agree.
On those, the judge agrees perfectly (kappa 1.0). Scoring against A alone gave kappa 0.89 and hid
something that matters: the judge made the same mistake as B on j20, a clear factual error. So: fix the
rubric for j16-type cases, treat j20-type disagreements as errors, and report judge agreement on the
consensus labels, noting the items where the judge sided with a human who was wrong.